# Poisson conformal calibration

The calibration score determines how a split-conformal band adapts to a fitted count mean. This study compares absolute residuals, Poisson-scale normalized residuals and Poisson central-quantile scores, holding each mean model fixed across scores. Every method is evaluated both by real interval length and by the number of nonnegative integers in its prediction set.

The audit of the baseline Poisson panel (seeds 6311000–6311049) reproduces every prediction exactly. Its supplied Poisson interval has 4.43% greater real length (paired Monte Carlo SE 1.52 percentage points) than the eligible real-width leader (the method with the shortest real intervals among those whose mean coverage is within 0.02 of 0.90), while containing 5.35% fewer counts (SE 0.45 percentage points) than the eligible count-size leader (the method with the fewest possible counts among the same eligible methods). The two metrics rank these prediction sets differently. Fresh confirmation uses 50 independent draws per count rate, with supplied-specification and data-only tracks. The headline table for the original-rate panel reports all methods; the low- and high-rate extensions follow separately.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import polars as pl
from great_tables import GT, md

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(root / "python"))
sys.path.insert(0, str(root))
import mboostrs as mb
from benches.summarize_poisson_conformal import load, validate
from benches.study_poisson_conformal import draw

directory = root / "docs/book/benchmarks/poisson_conformal"
summary = json.loads((directory / "summary.json").read_text())
validation = json.loads((directory / "validation.json").read_text())

In [2]:
def formatted(stat, digits=3, multiplier=1):
    if stat is None:
        return "unavailable"
    return f'{stat["mean"]*multiplier:.{digits}f} ({stat["se"]*multiplier:.{digits}f})'

def table_rows(phase, panel):
    selected = [r for r in summary if r["phase"] == phase and r["panel"] == panel]
    best = {}
    for row in selected:
        track = "Supplied specification" if row["method"].startswith("known") else "Data only"
        if row["coverage_eligible"]:
            best[track] = min(best.get(track, float("inf")), row["count_size"]["mean"])
    rows = []
    for row in selected:
        track = "Supplied specification" if row["method"].startswith("known") else "Data only"
        size = formatted(row["count_size"])
        if row["coverage_eligible"] and row["count_size"]["mean"] == best.get(track):
            size = "**"+size+"**"
        rows.append({"Information": track, "Method": row["method"].replace("_", " "),
                     "Draws": f'{row["n"]}/{row["n"]+row["failures"]}',
                     "Coverage %": formatted(row["coverage"], 2, 100), "Count size": size,
                     "Real length": formatted(row["width"]), "Interval score": formatted(row["interval_score"]),
                     "Point MSE": formatted(row["point_mse"])})
    return pl.DataFrame(rows)

In [3]:
def headline(phase, panel):
    return (GT(table_rows(phase, panel), groupname_col="Information")
        .tab_header(title=f"Poisson prediction sets: {panel} rate ({phase})",
                    subtitle="Nominal coverage 90%; count size, real length, interval score and MSE: smaller is better")
        .tab_spanner(label="Prediction-set efficiency", columns=["Count size", "Real length"])
        .tab_spanner(label="Real-endpoint and point scores", columns=["Interval score", "Point MSE"])
        .fmt_markdown(columns="Count size")
        .tab_source_note(md("Cells are mean (Monte Carlo SE) across independent draws. "
            "Each draw has 500 fitting, 500 calibration and 1000 independent test rows. "
            "Count size is the number of possible nonnegative counts; real length and interval score "
            "use count units, MSE uses squared count units. Point predictions are fitted means for "
            "mboostrs and midpoints for CQR. Bold marks the smallest count size within each information "
            "track among rows with coverage in [88%,92%]. The selection is retrospective and the SE "
            "does not adjust for that selection. All methods use the same support intersection.")))

headline("confirmation", "original")

GT(_tbl_data=shape: (11, 8)
┌─────────────┬─────────────┬───────┬─────────────┬────────────┬─────────┬────────────┬────────────┐
│ Information ┆ Method      ┆ Draws ┆ Coverage %  ┆ Count size ┆ Real    ┆ Interval   ┆ Point MSE  │
│ ---         ┆ ---         ┆ ---   ┆ ---         ┆ ---        ┆ length  ┆ score      ┆ ---        │
│ str         ┆ str         ┆ str   ┆ str         ┆ str        ┆ ---     ┆ ---        ┆ str        │
│             ┆             ┆       ┆             ┆            ┆ str     ┆ str        ┆            │
╞═════════════╪═════════════╪═══════╪═════════════╪════════════╪═════════╪════════════╪════════════╡
│ Data only   ┆ catboost    ┆ 50/50 ┆ 90.03       ┆ 3.957      ┆ 3.980   ┆ 5.522      ┆ 1.652      │
│             ┆ cqr         ┆       ┆ (0.24)      ┆ (0.021)    ┆ (0.032) ┆ (0.037)    ┆ (0.012)    │
│ Data only   ┆ catboost    ┆ 50/50 ┆ 90.05       ┆ 3.815      ┆ 4.152   ┆ 5.619      ┆ 1.688      │
│             ┆ split       ┆       ┆ (0.25)      ┆ (0.024)    ┆ (0.031) ┆ (0.026)    ┆ (0.014)    │
│ Data only   ┆ discovery   ┆ 50/50 ┆ 90.09       ┆ 3.765      ┆ 3.713   ┆ 5.386      ┆ 1.622      │
│             ┆ absolute    ┆       ┆ (0.29)      ┆ (0.035)    ┆ (0.043) ┆ (0.035)    ┆ (0.014)    │
│ Data only   ┆ discovery   ┆ 50/50 ┆ 90.06       ┆ **3.662    ┆ 3.506   ┆ 5.161      ┆ 1.622      │
│             ┆ poisson     ┆       ┆ (0.29)      ┆ (0.036)**  ┆ (0.044) ┆ (0.039)    ┆ (0.014)    │
│             ┆ normalized  ┆       ┆             ┆            ┆         ┆            ┆            │
│ Data only   ┆ discovery   ┆ 50/50 ┆ 96.88       ┆ 4.699      ┆ 3.719   ┆ 4.611      ┆ 1.622      │
│             ┆ poisson     ┆       ┆ (0.08)      ┆ (0.016)    ┆ (0.030) ┆ (0.038)    ┆ (0.014)    │
│             ┆ quantile    ┆       ┆             ┆            ┆         ┆            ┆            │
│ …           ┆ …           ┆ …     ┆ …           ┆ …          ┆ …       ┆ …          ┆ …          │
│ Supplied    ┆ known       ┆ 50/50 ┆ 89.76       ┆ **3.574    ┆ 3.504   ┆ 4.922      ┆ 1.483      │
│ specificati ┆ poisson     ┆       ┆ (0.25)      ┆ (0.022)**  ┆ (0.026) ┆ (0.021)    ┆ (0.010)    │
│ on          ┆ normalized  ┆       ┆             ┆            ┆         ┆            ┆            │
│ Supplied    ┆ known       ┆ 50/50 ┆ 97.06       ┆ 4.624      ┆ 3.624   ┆ 4.379      ┆ 1.483      │
│ specificati ┆ poisson     ┆       ┆ (0.07)      ┆ (0.012)    ┆ (0.012) ┆ (0.019)    ┆ (0.010)    │
│ on          ┆ quantile    ┆       ┆             ┆            ┆         ┆            ┆            │
│ Data only   ┆ lightgbm    ┆ 50/50 ┆ 89.88       ┆ 3.821      ┆ 3.668   ┆ 5.258      ┆ 1.638      │
│             ┆ cqr         ┆       ┆ (0.26)      ┆ (0.027)    ┆ (0.053) ┆ (0.033)    ┆ (0.013)    │
│ Data only   ┆ lightgbm    ┆ 50/50 ┆ 89.93       ┆ 3.867      ┆ 4.277   ┆ 5.757      ┆ 1.783      │
│             ┆ split       ┆       ┆ (0.22)      ┆ (0.022)    ┆ (0.029) ┆ (0.027)    ┆ (0.014)    │
│ Data only   ┆ sklearn cqr ┆ 50/50 ┆ 93.71       ┆ 4.148      ┆ 3.559   ┆ 4.687      ┆ 1.730      │
│             ┆             ┆       ┆ (0.27)      ┆ (0.032)    ┆ (0.025) ┆ (0.034)    ┆ (0.014)    │
└─────────────┴─────────────┴───────┴─────────────┴────────────┴─────────┴────────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x10d218bf0>, _boxhead=Boxhead([ColInfo(var='Information', type=<ColInfoTypeEnum.row_group: 3>, column_label='Information', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Draws', type=<ColInfoTypeEnum.default: 1>, column_label='Draws', column_align='left', column_width=None), ColInfo(var='Coverage %', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage %', column_align='left', column_width=None), ColInfo(var='Count size', type=<ColInfoTypeEnum.default: 1>, column_label='Count size', column_align='left', column_width=None), ColInfo(var='Real length', type=

## Calibration score functions

The default absolute score is $|y-\widehat\mu(x)|$, giving a constant-width real band. `score="poisson_normalized"` divides this residual by $\sqrt{\max(\widehat\mu(x),10^{-12})}$ and multiplies the conformal correction by that scale at prediction. `score="poisson_quantile"` calibrates the discrepancy outside the central Poisson quantile band. The Poisson mean fit is identical across these calls.

These are fixed score functions learned independently of calibration responses. The exchangeable-row marginal conformal guarantee does not require a correct Poisson variance assumption, but that assumption affects efficiency and conditional coverage. Quantile-score ties can make a discrete band conservative. Endpoints remain real-valued; count-support intersection is applied equally to every method during evaluation. The normalized correction is in standardized residual units, while the absolute and quantile corrections are in response units. The construction follows the split-conformal and locally adaptive score framework described by [Romano, Patterson and Candès (2019)](https://arxiv.org/abs/1905.03222).

In [4]:
from mboostrs.cvrisk import cv_weights

frame, _ = draw("original", 6361000)
future, _ = draw("original", 6411000)
weights = cv_weights(frame.height, type="kfold", B=2, seed=6361000)[:, 0]
training = frame[np.flatnonzero(weights > 0)]
calibration = frame[np.flatnonzero(weights == 0)]
model = mb.gamboost("y ~ bols(x1) + bols(x2)", training, family=mb.Poisson(),
                    control=mb.boost_control(100))
query = future.head(3).drop("y")
bands = {score: mb.conformal_interval(model, query, calibration=calibration, score=score)
         for score in ("absolute", "poisson_normalized", "poisson_quantile")}
for band in bands.values():
    np.testing.assert_array_equal(band.fitted_mean, model.predict(query, type="response"))

In [5]:
preview = []
for score, band in bands.items():
    for row, (mean, lower, upper) in enumerate(zip(band.fitted_mean, band.lower, band.upper, strict=True)):
        preview.append({"Score": score, "Row": row, "Fitted mean": mean,
                        "Lower": lower, "Upper": upper})
(GT(pl.DataFrame(preview))
 .tab_header(title="One fitted mean, three calibration scores",
             subtitle="Real endpoints in count units; the conditional mean remains unchanged")
 .fmt_number(columns=["Fitted mean", "Lower", "Upper"], decimals=4)
 .tab_source_note("The formula and 100-iteration budget are supplied in advance. "
                  "Calibration rows do not choose the family, formula, stopping or score."))

GT(_tbl_data=shape: (9, 5)
┌────────────────────┬─────┬─────────────┬───────────┬──────────┐
│ Score              ┆ Row ┆ Fitted mean ┆ Lower     ┆ Upper    │
│ ---                ┆ --- ┆ ---         ┆ ---       ┆ ---      │
│ str                ┆ i64 ┆ f64         ┆ f64       ┆ f64      │
╞════════════════════╪═════╪═════════════╪═══════════╪══════════╡
│ absolute           ┆ 0   ┆ 0.676652    ┆ -1.200591 ┆ 2.553895 │
│ absolute           ┆ 1   ┆ 1.64901     ┆ -0.228233 ┆ 3.526253 │
│ absolute           ┆ 2   ┆ 1.585949    ┆ -0.291294 ┆ 3.463193 │
│ poisson_normalized ┆ 0   ┆ 0.676652    ┆ -0.528648 ┆ 1.881952 │
│ poisson_normalized ┆ 1   ┆ 1.64901     ┆ -0.232576 ┆ 3.530596 │
│ poisson_normalized ┆ 2   ┆ 1.585949    ┆ -0.259309 ┆ 3.431207 │
│ poisson_quantile   ┆ 0   ┆ 0.676652    ┆ 0.0       ┆ 2.0      │
│ poisson_quantile   ┆ 1   ┆ 1.64901     ┆ 0.0       ┆ 4.0      │
│ poisson_quantile   ┆ 2   ┆ 1.585949    ┆ 0.0       ┆ 4.0      │
└────────────────────┴─────┴─────────────┴───────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x10cc3cbf0>, _boxhead=Boxhead([ColInfo(var='Score', type=<ColInfoTypeEnum.default: 1>, column_label='Score', column_align='left', column_width=None), ColInfo(var='Row', type=<ColInfoTypeEnum.default: 1>, column_label='Row', column_align='right', column_width=None), ColInfo(var='Fitted mean', type=<ColInfoTypeEnum.default: 1>, column_label='Fitted mean', column_align='right', column_width=None), ColInfo(var='Lower', type=<ColInfoTypeEnum.default: 1>, column_label='Lower', column_align='right', column_width=None), ColInfo(var='Upper', type=<ColInfoTypeEnum.default: 1>, column_label='Upper', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10b9171a0>, _spanners=Spanners([]), _heading=Heading(title='One fitted mean, three calibration scores', subtitle='Real endpoints in count units; the conditional mean remains unchanged', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10d270c80>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10d270aa0>, _source_notes=['The formula and 100-iteration budget are supplied in advance. Calibration rows do not choose the family, formula, stopping or score.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10d270590>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10d29ddf0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo

## Count support and set size

For a real interval $[l,u]$, the possible observed counts are $\{k\in\mathbb Z: k\ge0, l\le k\le u\}$. Its smallest and largest possible counts are $L=\max(0,\lceil l\rceil)$ and $U=\lfloor u\rfloor$. The set has size $\max(U-L+1,0)$. A nonempty singleton has size one and endpoint span zero. A nonempty real interval may contain no integer; it must remain an empty count set.

Intersection with the known support preserves the covered count observations exactly. Real length, length after intersection with the nonnegative real line, integer endpoint span and integer-set cardinality measure different objects. The audit keeps the baseline real-width score intact and adds the other estimands for every method.

In [6]:
audit_rows = [r for r in summary if r["phase"] == "audit"]
audit_table = pl.DataFrame([{"Method": r["method"],
    "Coverage %": formatted(r["coverage"], 2, 100), "Raw real length": formatted(r["width"]),
    "Nonnegative length": formatted(r["support_length"]), "Integer span": formatted(r["count_span"]),
    "Count size": formatted(r["count_size"])} for r in audit_rows])
(GT(audit_table)
 .tab_header(title="Support audit of the preceding 50 Poisson draws",
             subtitle="Mean (Monte Carlo SE); size is in possible counts, lengths and spans in count units")
 .tab_spanner(label="Different efficiency estimands", columns=["Raw real length", "Nonnegative length",
                                                              "Integer span", "Count size"])
 .tab_source_note("All 350 prediction digests and all raw metrics reproduce exactly. "
                  "Every observed count has identical coverage before and after support intersection. "
                  "These are retrospective draws, distinct from the fresh confirmation."))

GT(_tbl_data=shape: (7, 6)
┌────────────────┬──────────────┬─────────────────┬────────────────┬───────────────┬───────────────┐
│ Method         ┆ Coverage %   ┆ Raw real length ┆ Nonnegative    ┆ Integer span  ┆ Count size    │
│ ---            ┆ ---          ┆ ---             ┆ length         ┆ ---           ┆ ---           │
│ str            ┆ str          ┆ str             ┆ ---            ┆ str           ┆ str           │
│                ┆              ┆                 ┆ str            ┆               ┆               │
╞════════════════╪══════════════╪═════════════════╪════════════════╪═══════════════╪═══════════════╡
│ catboost_cqr   ┆ 90.21 (0.23) ┆ 3.995 (0.040)   ┆ 3.576 (0.025)  ┆ 2.967 (0.026) ┆ 3.963 (0.026) │
│ catboost_split ┆ 90.22 (0.23) ┆ 4.186 (0.032)   ┆ 3.442 (0.019)  ┆ 2.825 (0.022) ┆ 3.825 (0.022) │
│ discovery      ┆ 90.04 (0.24) ┆ 3.682 (0.032)   ┆ 3.295 (0.018)  ┆ 2.729 (0.025) ┆ 3.729 (0.025) │
│ known          ┆ 90.09 (0.25) ┆ 3.839 (0.025)   ┆ 3.278 (0.016)  ┆ 2.618 (0.019) ┆ 3.618 (0.019) │
│ lightgbm_cqr   ┆ 89.96 (0.27) ┆ 3.715 (0.060)   ┆ 3.386 (0.033)  ┆ 2.837 (0.027) ┆ 3.837 (0.027) │
│ lightgbm_split ┆ 90.14 (0.22) ┆ 4.324 (0.029)   ┆ 3.505 (0.017)  ┆ 2.886 (0.019) ┆ 3.886 (0.019) │
│ sklearn_cqr    ┆ 93.68 (0.23) ┆ 3.546 (0.023)   ┆ 3.545 (0.023)  ┆ 3.130 (0.031) ┆ 4.130 (0.031) │
└────────────────┴──────────────┴─────────────────┴────────────────┴───────────────┴───────────────┘, _body=<great_tables._gt_data.Body object at 0x10ccd5730>, _boxhead=Boxhead([ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Coverage %', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage %', column_align='left', column_width=None), ColInfo(var='Raw real length', type=<ColInfoTypeEnum.default: 1>, column_label='Raw real length', column_align='left', column_width=None), ColInfo(var='Nonnegative length', type=<ColInfoTypeEnum.default: 1>, column_label='Nonnegative length', column_align='left', column_width=None), ColInfo(var='Integer span', type=<ColInfoTypeEnum.default: 1>, column_label='Integer span', column_align='left', column_width=None), ColInfo(var='Count size', type=<ColInfoTypeEnum.default: 1>, column_label='Count size', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10d404ef0>, _spanners=Spanners([SpannerInfo(spanner_id='Different efficiency estimands', spanner_level=0, spanner_label='Different efficiency estimands', spanner_units=None, spanner_pattern=None, vars=['Raw real length', 'Nonnegative length', 'Integer span', 'Count size'], built=None)]), _heading=Heading(title='Support audit of the preceding 50 Poisson draws', subtitle='Mean (Monte Carlo SE); size is in possible counts, lengths and spans in count units', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10abacf80>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10d270f20>, _source_notes=['All 350 prediction digests and all raw metrics reproduce exactly. Every observed count has identical coverage before and after support intersection. These are retrospective draws, distinct from the fresh confirmation.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10d272f60>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'

## Independent confirmation

Development uses seeds 6361000–6361019; confirmation uses 6362000–6362049. They are disjoint from the baseline Poisson panel. At each rate the response is Poisson with mean $a\exp(0.3+0.6x_1-0.4x_2)$ and independent uniform[-1,1] predictors. The multiplier $a$ is 0.2, 1 or 10. Test draws use the development seed plus 50000. Panels share seed streams and are reported separately.

The supplied track has the Poisson family and linear log-mean formula, fixed at 100 iterations with rate 0.1. The data-only track calls `discover` on only the fitting rows and supplies no family or formula. Each selected model is held fixed across scores. CQR (conformalized quantile regression) fits quantile models for the lower and upper ends of the interval and widens them by a conformal correction. The five untuned controls are CatBoost split/CQR, LightGBM split/CQR and sklearn gradient-boosting CQR, with the settings fixed in the [protocol](poisson_conformal/PROTOCOL.md). All use the same actual 500/500 Rust-generated partition.

The following table gives paired differences against the absolute score from the same mean fit. A change in point MSE must be zero when only the calibration score changes. Score options are not selected using confirmation outcomes.

In [7]:
paired = []
for row in summary:
    if row["phase"] == "confirmation" and "paired_to_absolute" in row and not row["method"].endswith("absolute"):
        delta = row["paired_to_absolute"]
        paired.append({"Rate": row["panel"], "Method": row["method"],
            "Count-size change": formatted(delta["count_size"]),
            "Coverage change, pp": formatted(delta["coverage"], 3, 100),
            "Point-MSE change": formatted(delta["point_mse"], 6)})
(GT(pl.DataFrame(paired))
 .tab_header(title="Paired score changes with each mean fit held fixed",
             subtitle="New score minus absolute score; mean (Monte Carlo SE) over 50 matched draws")
 .tab_source_note("A negative count-size change means fewer possible counts. "
                  "Coverage changes are percentage points. Each rate is a separate experiment."))

GT(_tbl_data=shape: (12, 5)
┌──────────┬───────────────────────┬───────────────────┬─────────────────────┬─────────────────────┐
│ Rate     ┆ Method                ┆ Count-size change ┆ Coverage change, pp ┆ Point-MSE change    │
│ ---      ┆ ---                   ┆ ---               ┆ ---                 ┆ ---                 │
│ str      ┆ str                   ┆ str               ┆ str                 ┆ str                 │
╞══════════╪═══════════════════════╪═══════════════════╪═════════════════════╪═════════════════════╡
│ high     ┆ discovery_poisson_nor ┆ 0.018 (0.060)     ┆ 0.108 (0.146)       ┆ 0.000000 (0.000000) │
│          ┆ malized               ┆                   ┆                     ┆                     │
│ high     ┆ discovery_poisson_qua ┆ 1.140 (0.114)     ┆ 2.042 (0.223)       ┆ 0.000000 (0.000000) │
│          ┆ ntile                 ┆                   ┆                     ┆                     │
│ high     ┆ known_poisson_normali ┆ -0.680 (0.124)    ┆ 0.022 (0.149)       ┆ 0.000000 (0.000000) │
│          ┆ zed                   ┆                   ┆                     ┆                     │
│ high     ┆ known_poisson_quantil ┆ 0.211 (0.114)     ┆ 1.824 (0.191)       ┆ 0.000000 (0.000000) │
│          ┆ e                     ┆                   ┆                     ┆                     │
│ low      ┆ discovery_poisson_nor ┆ 0.000 (0.000)     ┆ 0.000 (0.000)       ┆ 0.000000 (0.000000) │
│          ┆ malized               ┆                   ┆                     ┆                     │
│ …        ┆ …                     ┆ …                 ┆ …                   ┆ …                   │
│ low      ┆ known_poisson_quantil ┆ 0.655 (0.015)     ┆ 7.512 (0.199)       ┆ 0.000000 (0.000000) │
│          ┆ e                     ┆                   ┆                     ┆                     │
│ original ┆ discovery_poisson_nor ┆ -0.103 (0.014)    ┆ -0.028 (0.139)      ┆ 0.000000 (0.000000) │
│          ┆ malized               ┆                   ┆                     ┆                     │
│ original ┆ discovery_poisson_qua ┆ 0.934 (0.033)     ┆ 6.786 (0.288)       ┆ 0.000000 (0.000000) │
│          ┆ ntile                 ┆                   ┆                     ┆                     │
│ original ┆ known_poisson_normali ┆ -0.036 (0.014)    ┆ -0.148 (0.162)      ┆ 0.000000 (0.000000) │
│          ┆ zed                   ┆                   ┆                     ┆                     │
│ original ┆ known_poisson_quantil ┆ 1.014 (0.017)     ┆ 7.150 (0.224)       ┆ 0.000000 (0.000000) │
│          ┆ e                     ┆                   ┆                     ┆                     │
└──────────┴───────────────────────┴───────────────────┴─────────────────────┴─────────────────────┘, _body=<great_tables._gt_data.Body object at 0x109b69040>, _boxhead=Boxhead([ColInfo(var='Rate', type=<ColInfoTypeEnum.default: 1>, column_label='Rate', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Count-size change', type=<ColInfoTypeEnum.default: 1>, column_label='Count-size change', column_align='left', column_width=None), ColInfo(var='Coverage change, pp', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage change, pp', column_align='left', column_width=None), ColInfo(var='Point-MSE change', type=<ColInfoTypeEnum.default: 1>, column_label='Point-MSE change', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10d404f20>, _spanners=Spanners([]), _heading=Heading(title='Paired score changes with each mean fit held fixed', subtitle='New score minus absolute score; mean (Monte Carlo SE) over 50 matched draws', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10d29ea50>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10d29dfa0>, _source_notes=['A negative count-size change means fewer possible

In [8]:
gaps = []
for row in summary:
    if row["phase"] == "confirmation" and row["method"].startswith(("known_", "discovery_")):
        gaps.append({"Rate": row["panel"], "Method": row["method"],
            "Coverage eligible": row["coverage_eligible"],
            "Count-size leader": row["count_size_leader"] or "none",
            "Count-size excess %": formatted(row["count_size_gap_percent"], 2),
            "Real-width excess %": formatted(row["width_gap_percent"], 2)})
(GT(pl.DataFrame(gaps))
 .tab_header(title="Paired excess over eligible competitors",
             subtitle="Mean percentage excess (Monte Carlo SE in percentage points); smaller is better")
 .tab_source_note("Count-size and real-width leaders are chosen separately among the same five controls "
                  "with mean coverage in [88%,92%]. If none is eligible there is no leader. "
                  "A negative excess favours mboostrs. Leader choice is retrospective; no selection adjustment "
                  "is included. A candidate outside the coverage window cannot claim leadership."))

GT(_tbl_data=shape: (18, 6)
┌──────────┬────────────────────┬──────────┬────────────────────┬──────────────┬───────────────────┐
│ Rate     ┆ Method             ┆ Coverage ┆ Count-size leader  ┆ Count-size   ┆ Real-width excess │
│ ---      ┆ ---                ┆ eligible ┆ ---                ┆ excess %     ┆ %                 │
│ str      ┆ str                ┆ ---      ┆ str                ┆ ---          ┆ ---               │
│          ┆                    ┆ bool     ┆                    ┆ str          ┆ str               │
╞══════════╪════════════════════╪══════════╪════════════════════╪══════════════╪═══════════════════╡
│ high     ┆ discovery_absolute ┆ true     ┆ lightgbm_cqr       ┆ 24.87 (3.42) ┆ 24.96 (3.42)      │
│ high     ┆ discovery_poisson_ ┆ true     ┆ lightgbm_cqr       ┆ 25.01 (3.38) ┆ 25.09 (3.37)      │
│          ┆ normalized         ┆          ┆                    ┆              ┆                   │
│ high     ┆ discovery_poisson_ ┆ false    ┆ lightgbm_cqr       ┆ 33.40 (3.56) ┆ 25.99 (3.54)      │
│          ┆ quantile           ┆          ┆                    ┆              ┆                   │
│ high     ┆ known_absolute     ┆ true     ┆ lightgbm_cqr       ┆ 9.98 (3.32)  ┆ 10.72 (3.46)      │
│ high     ┆ known_poisson_norm ┆ true     ┆ lightgbm_cqr       ┆ 4.90 (2.66)  ┆ 4.97 (2.65)       │
│          ┆ alized             ┆          ┆                    ┆              ┆                   │
│ …        ┆ …                  ┆ …        ┆ …                  ┆ …            ┆ …                 │
│ original ┆ discovery_poisson_ ┆ true     ┆ catboost_split     ┆ -3.93 (0.91) ┆ -3.67 (1.52)      │
│          ┆ normalized         ┆          ┆                    ┆              ┆                   │
│ original ┆ discovery_poisson_ ┆ false    ┆ catboost_split     ┆ 23.37 (0.74) ┆ 2.26 (1.48)       │
│          ┆ quantile           ┆          ┆                    ┆              ┆                   │
│ original ┆ known_absolute     ┆ true     ┆ catboost_split     ┆ -5.30 (0.41) ┆ 4.81 (1.34)       │
│ original ┆ known_poisson_norm ┆ true     ┆ catboost_split     ┆ -6.23 (0.51) ┆ -3.74 (1.25)      │
│          ┆ alized             ┆          ┆                    ┆              ┆                   │
│ original ┆ known_poisson_quan ┆ false    ┆ catboost_split     ┆ 21.41 (0.69) ┆ -0.28 (1.37)      │
│          ┆ tile               ┆          ┆                    ┆              ┆                   │
└──────────┴────────────────────┴──────────┴────────────────────┴──────────────┴───────────────────┘, _body=<great_tables._gt_data.Body object at 0x109b68fe0>, _boxhead=Boxhead([ColInfo(var='Rate', type=<ColInfoTypeEnum.default: 1>, column_label='Rate', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Coverage eligible', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage eligible', column_align='center', column_width=None), ColInfo(var='Count-size leader', type=<ColInfoTypeEnum.default: 1>, column_label='Count-size leader', column_align='left', column_width=None), ColInfo(var='Count-size excess %', type=<ColInfoTypeEnum.default: 1>, column_label='Count-size excess %', column_align='left', column_width=None), ColInfo(var='Real-width excess %', type=<ColInfoTypeEnum.default: 1>, column_label='Real-width excess %', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10d29dbe0>, _spanners=Spanners([]), _heading=Heading(title='Paired excess over eligible competitors', subtitle='Mean percentage excess (Monte Carlo SE in percentage points); smaller is better', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10d4068a0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10d4067b0>, _source_notes=['Count-size and real-width leaders are chosen separately among the same five controls with mean coverage in 

## Rate extensions

The low-rate panel examines intervals with few possible counts, where ties and zero responses matter most. The high-rate panel examines a larger range of Poisson variances. Coverage, efficiency and uncertainty are read within each panel; the independent-draw count is 50 per rate, rather than 150 pooled draws.

In [9]:
headline("confirmation", "low")

GT(_tbl_data=shape: (11, 8)
┌─────────────┬─────────────┬───────┬─────────────┬────────────┬─────────┬────────────┬────────────┐
│ Information ┆ Method      ┆ Draws ┆ Coverage %  ┆ Count size ┆ Real    ┆ Interval   ┆ Point MSE  │
│ ---         ┆ ---         ┆ ---   ┆ ---         ┆ ---        ┆ length  ┆ score      ┆ ---        │
│ str         ┆ str         ┆ str   ┆ str         ┆ str        ┆ ---     ┆ ---        ┆ str        │
│             ┆             ┆       ┆             ┆            ┆ str     ┆ str        ┆            │
╞═════════════╪═════════════╪═══════╪═════════════╪════════════╪═════════╪════════════╪════════════╡
│ Data only   ┆ catboost    ┆ 50/50 ┆ 90.19       ┆ 1.829      ┆ 1.467   ┆ 2.406      ┆ 0.398      │
│             ┆ cqr         ┆       ┆ (0.20)      ┆ (0.009)    ┆ (0.017) ┆ (0.026)    ┆ (0.004)    │
│ Data only   ┆ catboost    ┆ 50/50 ┆ 90.05       ┆ 1.743      ┆ 1.754   ┆ 2.625      ┆ 0.343      │
│             ┆ split       ┆       ┆ (0.19)      ┆ (0.009)    ┆ (0.008) ┆ (0.020)    ┆ (0.004)    │
│ Data only   ┆ discovery   ┆ 50/50 ┆ 91.76       ┆ **1.728    ┆ 1.449   ┆ 2.338      ┆ 0.306      │
│             ┆ absolute    ┆       ┆ (0.38)      ┆ (0.024)**  ┆ (0.007) ┆ (0.025)    ┆ (0.004)    │
│ Data only   ┆ discovery   ┆ 50/50 ┆ 91.76       ┆ **1.728    ┆ 1.500   ┆ 2.373      ┆ 0.306      │
│             ┆ poisson     ┆       ┆ (0.38)      ┆ (0.024)**  ┆ (0.012) ┆ (0.025)    ┆ (0.004)    │
│             ┆ normalized  ┆       ┆             ┆            ┆         ┆            ┆            │
│ Data only   ┆ discovery   ┆ 50/50 ┆ 96.72       ┆ 2.120      ┆ 1.120   ┆ 1.861      ┆ 0.306      │
│             ┆ poisson     ┆       ┆ (0.14)      ┆ (0.019)    ┆ (0.019) ┆ (0.022)    ┆ (0.004)    │
│             ┆ quantile    ┆       ┆             ┆            ┆         ┆            ┆            │
│ …           ┆ …           ┆ …     ┆ …           ┆ …          ┆ …       ┆ …          ┆ …          │
│ Supplied    ┆ known       ┆ 50/50 ┆ 90.21       ┆ **1.609    ┆ 1.602   ┆ 2.439      ┆ 0.297      │
│ specificati ┆ poisson     ┆       ┆ (0.20)      ┆ (0.010)**  ┆ (0.014) ┆ (0.021)    ┆ (0.004)    │
│ on          ┆ normalized  ┆       ┆             ┆            ┆         ┆            ┆            │
│ Supplied    ┆ known       ┆ 50/50 ┆ 97.73       ┆ 2.265      ┆ 1.265   ┆ 1.773      ┆ 0.297      │
│ specificati ┆ poisson     ┆       ┆ (0.11)      ┆ (0.014)    ┆ (0.014) ┆ (0.019)    ┆ (0.004)    │
│ on          ┆ quantile    ┆       ┆             ┆            ┆         ┆            ┆            │
│ Data only   ┆ lightgbm    ┆ 50/50 ┆ 94.62       ┆ 1.972      ┆ 1.241   ┆ 1.961      ┆ 0.441      │
│             ┆ cqr         ┆       ┆ (0.35)      ┆ (0.018)    ┆ (0.016) ┆ (0.026)    ┆ (0.007)    │
│ Data only   ┆ lightgbm    ┆ 50/50 ┆ 90.06       ┆ 1.759      ┆ 1.855   ┆ 2.667      ┆ 0.358      │
│             ┆ split       ┆       ┆ (0.20)      ┆ (0.009)    ┆ (0.010) ┆ (0.017)    ┆ (0.003)    │
│ Data only   ┆ sklearn cqr ┆ 50/50 ┆ 95.24       ┆ 1.975      ┆ 1.158   ┆ 1.907      ┆ 0.413      │
│             ┆             ┆       ┆ (0.21)      ┆ (0.019)    ┆ (0.024) ┆ (0.028)    ┆ (0.009)    │
└─────────────┴─────────────┴───────┴─────────────┴────────────┴─────────┴────────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x10d29deb0>, _boxhead=Boxhead([ColInfo(var='Information', type=<ColInfoTypeEnum.row_group: 3>, column_label='Information', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Draws', type=<ColInfoTypeEnum.default: 1>, column_label='Draws', column_align='left', column_width=None), ColInfo(var='Coverage %', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage %', column_align='left', column_width=None), ColInfo(var='Count size', type=<ColInfoTypeEnum.default: 1>, column_label='Count size', column_align='left', column_width=None), ColInfo(var='Real length', type=

In [10]:
headline("confirmation", "high")

GT(_tbl_data=shape: (11, 8)
┌─────────────┬─────────────┬───────┬─────────────┬────────────┬─────────┬────────────┬────────────┐
│ Information ┆ Method      ┆ Draws ┆ Coverage %  ┆ Count size ┆ Real    ┆ Interval   ┆ Point MSE  │
│ ---         ┆ ---         ┆ ---   ┆ ---         ┆ ---        ┆ length  ┆ score      ┆ ---        │
│ str         ┆ str         ┆ str   ┆ str         ┆ str        ┆ ---     ┆ ---        ┆ str        │
│             ┆             ┆       ┆             ┆            ┆ str     ┆ str        ┆            │
╞═════════════╪═════════════╪═══════╪═════════════╪════════════╪═════════╪════════════╪════════════╡
│ Data only   ┆ catboost    ┆ 50/50 ┆ 90.22       ┆ 13.804     ┆ 13.803  ┆ 18.091     ┆ 16.283     │
│             ┆ cqr         ┆       ┆ (0.25)      ┆ (0.092)    ┆ (0.092) ┆ (0.097)    ┆ (0.118)    │
│ Data only   ┆ catboost    ┆ 50/50 ┆ 89.89       ┆ 13.435     ┆ 13.461  ┆ 17.808     ┆ 17.097     │
│             ┆ split       ┆       ┆ (0.25)      ┆ (0.097)    ┆ (0.099) ┆ (0.088)    ┆ (0.124)    │
│ Data only   ┆ discovery   ┆ 50/50 ┆ 90.27       ┆ 16.612     ┆ 16.611  ┆ 22.213     ┆ 28.747     │
│             ┆ absolute    ┆       ┆ (0.23)      ┆ (0.427)    ┆ (0.427) ┆ (0.603)    ┆ (1.514)    │
│ Data only   ┆ discovery   ┆ 50/50 ┆ 90.38       ┆ 16.629     ┆ 16.628  ┆ 21.406     ┆ 28.747     │
│             ┆ poisson     ┆       ┆ (0.22)      ┆ (0.420)    ┆ (0.420) ┆ (0.616)    ┆ (1.514)    │
│             ┆ normalized  ┆       ┆             ┆            ┆         ┆            ┆            │
│ Data only   ┆ discovery   ┆ 50/50 ┆ 92.31       ┆ 17.752     ┆ 16.752  ┆ 21.430     ┆ 28.747     │
│             ┆ poisson     ┆       ┆ (0.27)      ┆ (0.447)    ┆ (0.447) ┆ (0.612)    ┆ (1.514)    │
│             ┆ quantile    ┆       ┆             ┆            ┆         ┆            ┆            │
│ …           ┆ …           ┆ …     ┆ …           ┆ …          ┆ …       ┆ …          ┆ …          │
│ Supplied    ┆ known       ┆ 50/50 ┆ 89.99       ┆ **13.986   ┆ 13.985  ┆ 18.233     ┆ 21.240     │
│ specificati ┆ poisson     ┆       ┆ (0.23)      ┆ (0.356)**  ┆ (0.355) ┆ (0.540)    ┆ (1.321)    │
│ on          ┆ normalized  ┆       ┆             ┆            ┆         ┆            ┆            │
│ Supplied    ┆ known       ┆ 50/50 ┆ 91.79       ┆ 14.877     ┆ 13.890  ┆ 18.281     ┆ 21.240     │
│ specificati ┆ poisson     ┆       ┆ (0.21)      ┆ (0.395)    ┆ (0.400) ┆ (0.580)    ┆ (1.321)    │
│ on          ┆ quantile    ┆       ┆             ┆            ┆         ┆            ┆            │
│ Data only   ┆ lightgbm    ┆ 50/50 ┆ 90.13       ┆ **13.345   ┆ 13.334  ┆ 16.975     ┆ 16.389     │
│             ┆ cqr         ┆       ┆ (0.25)      ┆ (0.083)**  ┆ (0.083) ┆ (0.084)    ┆ (0.130)    │
│ Data only   ┆ lightgbm    ┆ 50/50 ┆ 89.98       ┆ 13.904     ┆ 13.945  ┆ 18.279     ┆ 18.140     │
│             ┆ split       ┆       ┆ (0.27)      ┆ (0.109)    ┆ (0.113) ┆ (0.104)    ┆ (0.154)    │
│ Data only   ┆ sklearn cqr ┆ 50/50 ┆ 90.26       ┆ 13.661     ┆ 13.659  ┆ 17.415     ┆ 17.299     │
│             ┆             ┆       ┆ (0.26)      ┆ (0.088)    ┆ (0.088) ┆ (0.094)    ┆ (0.167)    │
└─────────────┴─────────────┴───────┴─────────────┴────────────┴─────────┴────────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x10d407080>, _boxhead=Boxhead([ColInfo(var='Information', type=<ColInfoTypeEnum.row_group: 3>, column_label='Information', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Draws', type=<ColInfoTypeEnum.default: 1>, column_label='Draws', column_align='left', column_width=None), ColInfo(var='Coverage %', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage %', column_align='left', column_width=None), ColInfo(var='Count size', type=<ColInfoTypeEnum.default: 1>, column_label='Count size', column_align='left', column_width=None), ColInfo(var='Real length', type=

## Coverage by conditional mean

The following groups are the bottom and top thirds of the true conditional mean on the independent test rows. These evaluator-only groups are fixed by the generating mechanism and do not enter model or score selection. Coverage in these broad groups describes heterogeneity; the marginal conformal guarantee does not provide a conditional guarantee for either third.

In [11]:
conditional = []
for row in summary:
    if row["phase"] == "confirmation":
        conditional.append({"Rate": row["panel"], "Method": row["method"],
                            "Low mean, %": formatted(row["coverage_low"], 2, 100),
                            "High mean, %": formatted(row["coverage_high"], 2, 100)})
(GT(pl.DataFrame(conditional), groupname_col="Rate")
 .tab_header(title="Coverage within true-mean thirds",
             subtitle="Mean coverage % (Monte Carlo SE in percentage points); nominal 90%")
 .tab_source_note("Thirds are defined separately on each independent test draw. "
                  "Each rate has 50 independent draws; rates sharing streams are not pooled. "
                  "The evaluator's true mean is unavailable to the fitting procedures."))

GT(_tbl_data=shape: (33, 4)
┌──────────┬──────────────────────────────┬──────────────┬──────────────┐
│ Rate     ┆ Method                       ┆ Low mean, %  ┆ High mean, % │
│ ---      ┆ ---                          ┆ ---          ┆ ---          │
│ str      ┆ str                          ┆ str          ┆ str          │
╞══════════╪══════════════════════════════╪══════════════╪══════════════╡
│ high     ┆ catboost_cqr                 ┆ 92.01 (0.38) ┆ 86.21 (0.37) │
│ high     ┆ catboost_split               ┆ 96.53 (0.17) ┆ 81.32 (0.42) │
│ high     ┆ discovery_absolute           ┆ 95.56 (0.30) ┆ 79.50 (0.50) │
│ high     ┆ discovery_poisson_normalized ┆ 91.63 (0.34) ┆ 83.81 (0.71) │
│ high     ┆ discovery_poisson_quantile   ┆ 93.95 (0.38) ┆ 86.15 (0.68) │
│ …        ┆ …                            ┆ …            ┆ …            │
│ original ┆ known_poisson_normalized     ┆ 91.19 (0.24) ┆ 87.04 (0.53) │
│ original ┆ known_poisson_quantile       ┆ 97.63 (0.11) ┆ 96.39 (0.17) │
│ original ┆ lightgbm_cqr                 ┆ 93.22 (0.38) ┆ 87.09 (0.52) │
│ original ┆ lightgbm_split               ┆ 96.32 (0.16) ┆ 82.11 (0.40) │
│ original ┆ sklearn_cqr                  ┆ 96.01 (0.43) ┆ 90.56 (0.39) │
└──────────┴──────────────────────────────┴──────────────┴──────────────┘, _body=<great_tables._gt_data.Body object at 0x10d407050>, _boxhead=Boxhead([ColInfo(var='Rate', type=<ColInfoTypeEnum.row_group: 3>, column_label='Rate', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Low mean, %', type=<ColInfoTypeEnum.default: 1>, column_label='Low mean, %', column_align='left', column_width=None), ColInfo(var='High mean, %', type=<ColInfoTypeEnum.default: 1>, column_label='High mean, %', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10d405280>, _spanners=Spanners([]), _heading=Heading(title='Coverage within true-mean thirds', subtitle='Mean coverage % (Monte Carlo SE in percentage points); nominal 90%', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10d445070>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10d444f50>, _source_notes=["Thirds are defined separately on each independent test draw. Each rate has 50 independent draws; rates sharing streams are not pooled. The evaluator's true mean is unavailable to the fitting procedures."], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10d444f20>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='ta

## Cost and failures

A mean-model fit is shared by its three score variants. The following table reports that shared fit/selection cost separately from each calibration-and-prediction stage, which also includes evaluation and array materialization. Competitor elapsed time includes its estimator fits and calibration. These local timings describe whole workflows and do not estimate controlled implementation speed ratios. Every method uses one computational thread.

The archives retain all warnings and failures. Unsupported data-only models are retained as failures rather than replaced with the supplied Poisson model. The rows below also verify matched partitions, fixed mean predictions across scores, record counts and lossless archive checksums.

In [12]:
from benches.summarize_poisson_conformal import mean_se

records, _ = load("confirmation")
cost_rows = []
for method in sorted({r["method"] for r in records}):
    group = [r for r in records if r["panel"] == "original" and r["method"] == method and r["status"] == "ok"]
    fit_seconds = [r["details"]["fit_elapsed"] for r in group if "fit_elapsed" in r["details"]]
    cost_rows.append({"Method": method, "Shared mean fit/selection, s":
        formatted(mean_se(fit_seconds)) if fit_seconds else "included in control total",
        "Band/evaluation or control total, s": formatted(mean_se([r["elapsed"] for r in group]), 4)})
(GT(pl.DataFrame(cost_rows))
 .tab_header(title="Original-rate workflow cost", subtitle="Elapsed seconds: mean (Monte Carlo SE)")
 .tab_source_note("Mean fit/selection is shared across three scores; each band stage is timed separately, including evaluation and array materialization. "
                  "Control totals include their fits and calibration. These are one-thread local workflow "
                  "measurements, not controlled implementation speed ratios."))

GT(_tbl_data=shape: (11, 3)
┌──────────────────────────────┬──────────────────────────────┬─────────────────────────────────┐
│ Method                       ┆ Shared mean fit/selection, s ┆ Band/evaluation or control tot… │
│ ---                          ┆ ---                          ┆ ---                             │
│ str                          ┆ str                          ┆ str                             │
╞══════════════════════════════╪══════════════════════════════╪═════════════════════════════════╡
│ catboost_cqr                 ┆ included in control total    ┆ 0.3160 (0.0022)                 │
│ catboost_split               ┆ included in control total    ┆ 0.1555 (0.0013)                 │
│ discovery_absolute           ┆ 0.143 (0.003)                ┆ 0.0010 (0.0000)                 │
│ discovery_poisson_normalized ┆ 0.143 (0.003)                ┆ 0.0009 (0.0000)                 │
│ discovery_poisson_quantile   ┆ 0.143 (0.003)                ┆ 0.0013 (0.0000)                 │
│ …                            ┆ …                            ┆ …                               │
│ known_poisson_normalized     ┆ 0.001 (0.000)                ┆ 0.0007 (0.0000)                 │
│ known_poisson_quantile       ┆ 0.001 (0.000)                ┆ 0.0011 (0.0000)                 │
│ lightgbm_cqr                 ┆ included in control total    ┆ 0.0561 (0.0007)                 │
│ lightgbm_split               ┆ included in control total    ┆ 0.0408 (0.0003)                 │
│ sklearn_cqr                  ┆ included in control total    ┆ 0.2171 (0.0025)                 │
└──────────────────────────────┴──────────────────────────────┴─────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x10d404890>, _boxhead=Boxhead([ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Shared mean fit/selection, s', type=<ColInfoTypeEnum.default: 1>, column_label='Shared mean fit/selection, s', column_align='left', column_width=None), ColInfo(var='Band/evaluation or control total, s', type=<ColInfoTypeEnum.default: 1>, column_label='Band/evaluation or control total, s', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x109b69d00>, _spanners=Spanners([]), _heading=Heading(title='Original-rate workflow cost', subtitle='Elapsed seconds: mean (Monte Carlo SE)', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12a60afc0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12a60af60>, _source_notes=['Mean fit/selection is shared across three scores; each band stage is timed separately, including evaluation and array materialization. Control totals include their fits and calibration. These are one-thread local workflow measurements, not controlled implementation speed ratios.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x12a60ae70>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']

In [13]:
checks = []
for phase in ("audit", "development", "confirmation"):
    records, metadata = load(phase)
    verified = validate(phase, records)
    checks.append({"Phase": phase, "Records": verified["records"],
                   "Failures": verified["failures"], "Draws per rate": next(iter(verified["independent_seeds_per_panel"].values())),
                   "Support preserves coverage": verified["count_support_preserves_coverage"]})
    del records
(GT(pl.DataFrame(checks))
 .tab_header(title="Evidence validation", subtitle="All declared records and failures retained")
 .tab_source_note("Audit has one rate; development and confirmation have three separately reported rates. "
                  "Input hashes and all 1000 row identities agree within each draw."))

GT(_tbl_data=shape: (3, 5)
┌──────────────┬─────────┬──────────┬────────────────┬────────────────────────────┐
│ Phase        ┆ Records ┆ Failures ┆ Draws per rate ┆ Support preserves coverage │
│ ---          ┆ ---     ┆ ---      ┆ ---            ┆ ---                        │
│ str          ┆ i64     ┆ i64      ┆ i64            ┆ bool                       │
╞══════════════╪═════════╪══════════╪════════════════╪════════════════════════════╡
│ audit        ┆ 350     ┆ 0        ┆ 50             ┆ true                       │
│ development  ┆ 660     ┆ 0        ┆ 20             ┆ true                       │
│ confirmation ┆ 1650    ┆ 0        ┆ 50             ┆ true                       │
└──────────────┴─────────┴──────────┴────────────────┴────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x1213bcad0>, _boxhead=Boxhead([ColInfo(var='Phase', type=<ColInfoTypeEnum.default: 1>, column_label='Phase', column_align='left', column_width=None), ColInfo(var='Records', type=<ColInfoTypeEnum.default: 1>, column_label='Records', column_align='right', column_width=None), ColInfo(var='Failures', type=<ColInfoTypeEnum.default: 1>, column_label='Failures', column_align='right', column_width=None), ColInfo(var='Draws per rate', type=<ColInfoTypeEnum.default: 1>, column_label='Draws per rate', column_align='right', column_width=None), ColInfo(var='Support preserves coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Support preserves coverage', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1213bd460>, _spanners=Spanners([]), _heading=Heading(title='Evidence validation', subtitle='All declared records and failures retained', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1213bc860>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1213bd6d0>, _source_notes=['Audit has one rate; development and confirmation have three separately reported rates. Input hashes and all 1000 row identities agree within each draw.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1213bd6a0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border

## Interpretation and limitations

On the original-rate confirmation, normalized scoring gives supplied/discovery count-size excesses of -6.23% (MC SE 0.51 percentage points) and -3.93% (SE 0.91) relative to the eligible competitor. Coverage is 0.89764 and 0.90064. Real-width excesses are -3.74% (SE 1.25) and -3.67% (SE 1.52). Mean predictions and point MSE remain exactly unchanged across scores.

The high-rate extension retains a 25.01% discovery count-size excess (SE 3.38) with normalized scoring. All 50 discovery models choose Poisson and both mean terms, but 25 stop after one update. This motivates a separate investigation of the mean fit and stopping rule; the study does not isolate a cause. The supplied normalized model retains 4.90% excess (SE 2.66).

Central-quantile scores give supplied-model coverage 0.97732 and 0.97062 at low/original rates, outside the eligibility window of 0.88 to 0.92 (0.90 plus or minus 0.02). Their count-size excesses are +30.04% (SE 0.86) and +21.41% (SE 0.69). The tie-efficiency limitation remains open. These optional scores do not change the default, and no confirmation result chooses which score a caller should use.

Real-width excess and count-cardinality excess answer different questions. The audit identifies this distinction in the baseline Poisson panel; it does not change that panel's frozen primary metric. The new scores provide explicit alternatives with a common fitted mean. Neither a supplied formula nor the best point estimate in a confirmation panel establishes a universal one-percent bound.

The marginal conformal guarantee requires exchangeable rows and a score, model specification and stopping rule independent of calibration responses. It does not imply conditional coverage, effect-inference validity or suitability for clustered or time-ordered data. A Poisson working model may be inefficient under overdispersion or zero inflation; those distributions are outside these three Poisson panels. Discrete quantile ties can cause conservative coverage, which is reported without relaxing the eligibility rule.

Efficiency questions remain unresolved. A separate predictor-coding defect is open, and its split calibration has been corrected; CV+ (a conformal interval built from cross-validation errors) and CQR remain open. Their probes are distinct from the frozen competitor runs. The [protocol](poisson_conformal/PROTOCOL.md) states the methods and estimands fixed before implementation and measurement.

The [validation report](poisson_conformal/VALIDATION.md) describes the retained evidence and checks. The [runner](https://github.com/fcela/mboostrs/blob/main/benches/study_poisson_conformal.py) fixes the data, fitting and scoring procedures; the [summary script](https://github.com/fcela/mboostrs/blob/main/benches/summarize_poisson_conformal.py) records the numerical reporting rules.
